# Task: Arabic Aspect–Opinion Extraction from Reviews using CFG/PCFG + Dependency Parsing

> note: use Stanza.

In [22]:

data = [
    {
        "sentence": "الغرفة نظيفة",
        "gold_pairs": [("الغرفة", "نظيفة")]
    },
    {
        "sentence": "السرير مريح",
        "gold_pairs": [("السرير", "مريح")]
    },
    {
        "sentence": "خدمة الفندق ممتازة",
        "gold_pairs": [("خدمة الفندق", "ممتازة")]
    },
    {
        "sentence": "الموقع بعيد",
        "gold_pairs": [("الموقع", "بعيد")]
    },
    {
        "sentence": "السعر مرتفع",
        "gold_pairs": [("السعر", "مرتفع")]
    },
    {
        "sentence": "بطارية الهاتف ضعيفة",
        "gold_pairs": [("بطارية الهاتف", "ضعيفة")]
    },
    {
        "sentence": "الشاشة جميلة",
        "gold_pairs": [("الشاشة", "جميلة")]
    },
    {
        "sentence": "الطعام لذيذ",
        "gold_pairs": [("الطعام", "لذيذ")]
    },
    {
        "sentence": "الخدمة سريعة",
        "gold_pairs": [("الخدمة", "سريعة")]
    },
    {
        "sentence": "الغرفة نظيفة و السرير مريح",
        "gold_pairs": [("الغرفة", "نظيفة"), ("السرير", "مريح")]
    },
    {
        "sentence": "الموقع ممتاز لكن السعر مرتفع",
        "gold_pairs": [("الموقع", "ممتاز"), ("السعر", "مرتفع")]
    },
    {
        "sentence": "بطارية الهاتف ضعيفة و الشاشة جميلة",
        "gold_pairs": [("بطارية الهاتف", "ضعيفة"), ("الشاشة", "جميلة")]
    }
]

import pandas as pd
df = pd.DataFrame(data)
df


,sentence,gold_pairs
0,الغرفة نظيفة,"[(الغرفة, نظيفة)]"
1,السرير مريح,"[(السرير, مريح)]"
2,خدمة الفندق ممتازة,"[(خدمة الفندق, ممتازة)]"
3,الموقع بعيد,"[(الموقع, بعيد)]"
4,السعر مرتفع,"[(السعر, مرتفع)]"
5,بطارية الهاتف ضعيفة,"[(بطارية الهاتف, ضعيفة)]"
6,الشاشة جميلة,"[(الشاشة, جميلة)]"
7,الطعام لذيذ,"[(الطعام, لذيذ)]"
8,الخدمة سريعة,"[(الخدمة, سريعة)]"
9,الغرفة نظيفة و السرير مريح,"[(الغرفة, نظيفة), (السرير, مريح)]"


# Q1: Constituency parsing with a simple Arabic CFG.

Apply to following CFG to the constituency parser.

In [23]:
!pip install stanza


In [24]:
import stanza
stanza.download('ar')
nlp = stanza.Pipeline('ar')

INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading default packages for language: ar (Arabic) ...
INFO:stanza:File exists: /root/.cache/stanza/1.11.0/resources/ar/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: ar (Arabic):
| Processor | Package       |
-----------------------------
| tokenize  | padt          |
| mwt       | padt          |
| pos       | padt_charlm   |
| lemma     | padt_nocharlm |
| depparse  | padt_charlm   |
| ner       | aqmar_charlm  |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Loading: ner
INFO:stanza:Done loading processors!


In [25]:
import nltk
from nltk import CFG
grammar = CFG.fromstring("""
S -> CLAUSE
S -> CLAUSE CONJ CLAUSE

CLAUSE -> NP AP

NP -> NOUN
NP -> IDAFA

IDAFA -> N N

AP -> ADJ
AP -> ADV ADJ

CONJ -> 'و' | 'لكن'

NOUN -> 'الغرفة' | 'السرير' | 'الموقع' | 'السعر' | 'الشاشة' | 'الطعام' | 'الخدمة'
N -> 'خدمة' | 'الفندق' | 'بطارية' | 'الهاتف'

ADJ -> 'نظيفة' | 'مريح' | 'ممتازة' | 'بعيد' | 'مرتفع' | 'ضعيفة' | 'جميلة' | 'لذيذ' | 'سريعة' | 'ممتاز'
ADV -> 'جدا'
""")

In [26]:
from nltk.parse import RecursiveDescentParser

parser = RecursiveDescentParser(grammar)

# 3. Processing the Data
results = []
for item in data:
    sentence = item['sentence']

    # Use Stanza to get tokens correctly
    doc = nlp(sentence)
    tokens = [word.text for sent in doc.sentences for word in sent.words]

    # Parse using the CFG
    try:
        parses = list(parser.parse(tokens))
        tree = parses[0] if parses else "No parse found"
    except ValueError as e:
        tree = f"Error: {e}"

    results.append({"sentence": sentence, "tree": tree})

# 4. Display Results
df_results = pd.DataFrame(results)
print(df_results)

                              sentence  \
0                         الغرفة نظيفة   
1                          السرير مريح   
2                   خدمة الفندق ممتازة   
3                          الموقع بعيد   
4                          السعر مرتفع   
5                  بطارية الهاتف ضعيفة   
6                         الشاشة جميلة   
7                          الطعام لذيذ   
8                         الخدمة سريعة   
9           الغرفة نظيفة و السرير مريح   
10        الموقع ممتاز لكن السعر مرتفع   
11  بطارية الهاتف ضعيفة و الشاشة جميلة   

                                                 tree  
0                  [[[(NOUN الغرفة)], [(ADJ نظيفة)]]]  
1                   [[[(NOUN السرير)], [(ADJ مريح)]]]  
2   [[[(IDAFA (N خدمة) (N الفندق))], [(ADJ ممتازة)]]]  
3                   [[[(NOUN الموقع)], [(ADJ بعيد)]]]  
4                   [[[(NOUN السعر)], [(ADJ مرتفع)]]]  
5   [[[(IDAFA (N بطارية) (N الهاتف))], [(ADJ ضعيفة...  
6                  [[[(NOUN الشاشة)], [(ADJ جميلة)]]]  
7    

# Q2- Parse the following Arabic sentences.

In [27]:
examples = [
    "الغرفة نظيفة",
    "خدمة الفندق ممتازة",
    "الغرفة نظيفة و السرير مريح",
    "الموقع ممتاز لكن السعر مرتفع"
]


In [28]:
for sent in examples:
    # Tokenize with Stanza
    doc = nlp(sent)
    tokens = [word.text for sentence in doc.sentences for word in sentence.words]

    print(f"Sentence: {sent}")
    print(f"Tokens: {tokens}")

    # Attempt to Parse
    try:
        parses = list(parser.parse(tokens))
        if parses:
            for tree in parses:
                print("Tree structure:")
                tree.pretty_print() # Displays the ASCII tree
        else:
            print("No valid parse found for this sentence.")
    except ValueError as e:
        print(f"Error during parsing: {e}")

    print("-" * 30)

Sentence: الغرفة نظيفة
Tokens: ['الغرفة', 'نظيفة']
Tree structure:
         S         
         |          
       CLAUSE      
   ______|______    
  NP            AP 
  |             |   
 NOUN          ADJ 
  |             |   
الغرفة        نظيفة

------------------------------
Sentence: خدمة الفندق ممتازة
Tokens: ['خدمة', 'الفندق', 'ممتازة']
Tree structure:
             S          
             |           
           CLAUSE       
        _____|______     
       NP           |   
       |            |    
     IDAFA          AP  
  _____|_____       |    
 N           N     ADJ  
 |           |      |    
خدمة       الفندق ممتازة

------------------------------
Sentence: الغرفة نظيفة و السرير مريح
Tokens: ['الغرفة', 'نظيفة', 'و', 'السرير', 'مريح']
Tree structure:
                     S                     
          ___________|____________          
       CLAUSE        |          CLAUSE     
   ______|______     |      ______|_____    
  NP            AP   |     NP           A

[link text](https://)# Q3- Draw the parsing tree for the following sentence.

In [29]:
sent = "خدمة الفندق ممتازة"

In [30]:
parser = RecursiveDescentParser(grammar)

# 4. Tokenize using Stanza
doc = nlp(sent)
# Extracting text from the tokens
tokens = [word.text for sentence in doc.sentences for word in sentence.words]

# 5. Parse and Print
print(f"Tokens found by Stanza: {tokens}")
print("-" * 30)
for tree in parser.parse(tokens):
    tree.pretty_print()

Tokens found by Stanza: ['خدمة', 'الفندق', 'ممتازة']
------------------------------
             S          
             |           
           CLAUSE       
        _____|______     
       NP           |   
       |            |    
     IDAFA          AP  
  _____|_____       |    
 N           N     ADJ  
 |           |      |    
خدمة       الفندق ممتازة



# Q4- Apply the following PCFG:

In [31]:
import stanza
from nltk import PCFG
from nltk.parse import ViterbiParser

pcfg_grammar = PCFG.fromstring("""
S -> CLAUSE [0.55]
S -> CLAUSE PP [0.45]

CLAUSE -> NP AP [1.0]

NP -> IDAFA [0.60]
NP -> NOUN [0.40]

IDAFA -> N N [1.0]

AP -> ADJ [0.35]
AP -> ADJ PP [0.65]

PP -> P NP2 [1.0]

NP2 -> NOUN2 [0.35]
NP2 -> NOUN2 ADJ2 [0.65]

N -> 'خدمة' [0.50] | 'الفندق' [0.50]
NOUN -> 'الخدمة' [0.50] | 'الموقع' [0.50]
ADJ -> 'ممتازة' [1.0]
P -> 'في' [1.0]
NOUN2 -> 'الغرفة' [1.0]
ADJ2 -> 'الصغيرة' [1.0]
""")



In [32]:
parser = ViterbiParser(pcfg_grammar)

# Q5- Use this PCFG to parse the following sentence:

In [33]:
ambiguous_sentence = "خدمة الفندق ممتازة في الغرفة الصغيرة"

In [34]:
doc = nlp(ambiguous_sentence)
tokens = [word.text for sentence in doc.sentences for word in sentence.words]

for tree in parser.parse(tokens):
    print("Most Probable Parse Tree:")
    tree.pretty_print()
    print(f"Total Probability: {tree.prob()}")

Most Probable Parse Tree:
                    S                              
                    |                               
                  CLAUSE                           
        ____________|_________                      
       |                      AP                   
       |             _________|____                 
       NP           |              PP              
       |            |      ________|_____           
     IDAFA          |     |             NP2        
  _____|_____       |     |         _____|_____     
 N           N     ADJ    P      NOUN2        ADJ2 
 |           |      |     |        |           |    
خدمة       الفندق ممتازة  في     الغرفة     الصغيرة

Total Probability: 0.034856250000000005


# Q6- Use a pre-trained model to perform depenecy parsing for the sentences in the given dataset.

In [36]:
!python -m spacy download xx_ent_wiki_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 24.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('xx_ent_wiki_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [37]:
import spacy
import pandas as pd

nlp = spacy.load("xx_ent_wiki_sm")


In [38]:
for sentence in df["sentence"]:
    doc = nlp(sentence)

    print(f"\nSentence: {sentence}")
    print("-"*40)

    for token in doc:
        print(f"{token.text:15} -> head: {token.head.text:15} dep: {token.dep_}")



Sentence: الغرفة نظيفة
----------------------------------------
الغرفة          -> head: الغرفة          dep: 
نظيفة           -> head: نظيفة           dep: 

Sentence: السرير مريح
----------------------------------------
السرير          -> head: السرير          dep: 
مريح            -> head: مريح            dep: 

Sentence: خدمة الفندق ممتازة
----------------------------------------
خدمة            -> head: خدمة            dep: 
الفندق          -> head: الفندق          dep: 
ممتازة          -> head: ممتازة          dep: 

Sentence: الموقع بعيد
----------------------------------------
الموقع          -> head: الموقع          dep: 
بعيد            -> head: بعيد            dep: 

Sentence: السعر مرتفع
----------------------------------------
السعر           -> head: السعر           dep: 
مرتفع           -> head: مرتفع           dep: 

Sentence: بطارية الهاتف ضعيفة
----------------------------------------
بطارية          -> head: بطارية          dep: 
الهاتف          -> head: الهاتف     